# DiffuQR - module-space diffusion training (Phase 2)

Trains the main model: a conditional diffusion model over the **QR module grid** (analog bits, +1 dark / -1 light),
conditioned on the rectified blurry crop, with version-fixed modules (finders, timing, alignment, version info)
pinned during sampling. Backbone is the standard `diffusers.UNet2DModel`; loss is binary cross-entropy on module
bits. Library code lives in `diffuqr/` (`diffusion.py`, `modspace.py`, `qrcodec.py`); this notebook only orchestrates.

**Before running on Kaggle**
1. Settings: Accelerator **GPU T4 x2** (or P100), **Internet on**.
2. Add-ons -> Secrets: `HF_TOKEN` (needs read on the dataset repo, write on the model repo).
3. Run once with `SMOKE = True` (about 10 min: checks install, data, training, checkpoint push, evaluation).
   Then set `SMOKE = False` and run all.
4. If the session ends early, just run all again: training resumes from the last checkpoint on the Hub.

Data used: `calibration/*` and `modspace/real_*.npz` from the private dataset repo. Training data is
**synthetic only** (generated here); the real buggy frames are evaluation only.
Checkpoints go to the model repo only, never the dataset repo.

**Fine-tuning** (`INIT_FROM`): a new run (own `RUN_NAME`, own checkpoints) starts from another run's EMA weights
with a fresh optimiser and LR schedule; `SYNTH` widens the synthetic distribution. Current setting: continue
`modspace_diff_v1` (40k steps, versions 10-18 at 7-16 px/module) for 10k steps on versions 1-18 at 2-16 px/module,
because buggy02 (v3 at ~2.6 px/module) is outside the v1 training range.

In [ ]:
# ---- Configuration ----------------------------------------------------------
RUN_NAME = "modspace_diff_v1_ft_lowres"     # "modspace_diff_v1" = the 40k-step run from scratch
INIT_FROM = "modspace_diff_v1/last.pt"      # start a NEW run from this checkpoint's EMA weights; None = from scratch
SYNTH = dict(min_version=1, camera_ppm=(2.0, 16.0))   # ModuleSampleConfig overrides; {} = v1 run (v10-18, 7-16 px/module)
MODE = "diffusion"          # "diffusion" (main) | "direct" (ablation: same network, one-shot, no noise)
SMOKE = True                # True first: tiny end-to-end check. Then False for the real run.
GIT_REF = "main"            # branch / tag / commit of github.com/Rakshit-Sawarn-iitb/DiffuQR

DATASET_REPO = "RakshitAi/IITB-BuggyQR"           # private; read-only here
MODEL_REPO = "RakshitAi/IITB-BuggyQR-diffusion"   # checkpoints + results ONLY here

TRAIN_STEPS = 10_000        # v1 run: 40_000
BATCH = 32
LR = 1e-4                   # v1 run: 2e-4 (fine-tuning: lower peak, fresh cosine schedule)
WARMUP = 200                # v1 run: 500
EMA_DECAY = 0.999
POOL_TRAIN = 16_000         # pre-generated synthetic samples kept in RAM (~6 GB)
POOL_VAL = 512              # held-out synthetic codes (disjoint seeds)
P_FRESH = 0.1               # fraction of each batch generated fresh in DataLoader workers
SAMPLE_STEPS = 20           # DDIM steps at evaluation
K_LIST = (1, 2, 4, 8, 16)   # samples per frame for the decode-rate-vs-K curve
EVAL_EVERY = 2000
SAVE_EVERY = 1000           # local checkpoint (cheap)
PUSH_EVERY = 5000           # push to the Hub (409 MB each; repo history is squashed after each push)
LOG_EVERY = 100
MAX_HOURS = 11.0            # stop + checkpoint before Kaggle's 12 h limit
NUM_WORKERS = 2             # DataLoader workers; 2 keep up with the GPU (~0.8 s/step) and use less host RAM
REAL_LIMIT = None           # None = all aligned real frames
SEED = 0

if SMOKE:
    RUN_NAME += "_smoke"
    TRAIN_STEPS, BATCH, POOL_TRAIN, POOL_VAL = 40, 4, 32, 8
    EVAL_EVERY, SAVE_EVERY, PUSH_EVERY, LOG_EVERY = 20, 20, 40, 10
    SAMPLE_STEPS, K_LIST, REAL_LIMIT, P_FRESH = 4, (1, 2), 6, 0.25
print(f"RUN_NAME={RUN_NAME}  MODE={MODE}  SMOKE={SMOKE}")

In [ ]:
# ---- Environment: install the diffuqr library, paths, token -------------------
import os, sys, subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
if ON_KAGGLE:
    if "diffuqr" in sys.modules:
        raise RuntimeError("An older diffuqr is already imported in this kernel. "
                           "Run -> Restart & clear outputs, then run all cells again.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segno", "reedsolo", "diffusers"], check=True)
    # --force-reinstall: the version string stays 0.1.0 across commits, so pip would otherwise keep a stale copy.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", "--no-cache-dir",
                    f"git+https://github.com/Rakshit-Sawarn-iitb/DiffuQR@{GIT_REF}"], check=True)
    import json as _json
    from importlib.metadata import distribution
    _url = _json.loads(distribution("diffuqr").read_text("direct_url.json") or "{}")
    print("installed diffuqr from commit", _url.get("vcs_info", {}).get("commit_id"))
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    WORK = Path("/kaggle/working")
else:  # local smoke run from the repo root (or notebooks/)
    root = Path.cwd() if (Path.cwd() / "diffuqr").exists() else Path.cwd().parent
    sys.path.insert(0, str(root))
    from diffuqr.config import HF_TOKEN
    WORK = root / "artifacts" / "kaggle_local"
DATA = WORK / "data"
CKPT_DIR = WORK / "checkpoints" / RUN_NAME
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PUSH = ON_KAGGLE           # local runs never push
assert HF_TOKEN, "HF_TOKEN missing (Kaggle: Add-ons -> Secrets)"

import importlib.util
import diffuqr, torch, diffusers
assert importlib.util.find_spec("diffuqr.diffusion"), "installed diffuqr is too old (no diffuqr.diffusion): check GIT_REF"
print("diffuqr from", Path(diffuqr.__file__).parent, "| torch", torch.__version__, "| diffusers", diffusers.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU = torch.cuda.device_count()
print("device:", DEVICE, "| GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])

In [ ]:
# ---- Data from the private HF dataset (calibration + real eval only) ----------
from huggingface_hub import HfApi, snapshot_download

api = HfApi(token=HF_TOKEN)
assert api.repo_info(DATASET_REPO, repo_type="dataset").private, f"{DATASET_REPO} is public: stop and make it private"
snapshot_download(DATASET_REPO, repo_type="dataset", token=HF_TOKEN, local_dir=str(DATA),
                  allow_patterns=["calibration/*", "modspace/real_*.npz"])
BANK_PATH = str(DATA / "calibration" / "kernels.npz")
REAL_FILES = sorted((DATA / "modspace").glob("real_*.npz"))
print("kernel bank:", BANK_PATH, "| real eval files:", [p.name for p in REAL_FILES])

if PUSH:
    api.create_repo(MODEL_REPO, repo_type="model", private=True, exist_ok=True)
    assert api.repo_info(MODEL_REPO, repo_type="model").private, f"{MODEL_REPO} is public: stop and make it private"

In [ ]:
# ---- Synthetic pools (train + held-out val) and real eval items --------------
import json, time, numpy as np
from diffuqr.diffusion import generate_pool
from diffuqr.modspace import ModuleSampleConfig
from diffuqr.pipeline import contrast_stretch

SAMPLE_CFG = ModuleSampleConfig(**SYNTH)
print(f"synthetic: versions {SAMPLE_CFG.min_version}-{SAMPLE_CFG.max_version}, {SAMPLE_CFG.camera_ppm} px/module")
t0 = time.time()
gen_workers = NUM_WORKERS if ON_KAGGLE else 1   # Linux fork on Kaggle; avoid Windows spawn locally
train_pool = generate_pool(POOL_TRAIN, BANK_PATH, seed0=SEED * 10_000_000, workers=gen_workers, cfg=SAMPLE_CFG)
val_pool = generate_pool(POOL_VAL, BANK_PATH, seed0=SEED * 10_000_000 + 9_000_000, workers=gen_workers,
                         cfg=SAMPLE_CFG)
print(f"pools: train {len(train_pool)}, val {len(val_pool)} in {time.time() - t0:.0f}s")
vers = np.array([p[2] for p in train_pool])
print("train versions:", {int(v): int((vers == v).sum()) for v in np.unique(vers)})

kl = np.array([p[4] for p in val_pool])
cuts = np.percentile(kl, [100 / 3, 200 / 3])
val_items = [{"input": p[0], "grid": p[1], "version": p[2], "payload": p[3],
              "split": "synth-val v1-9" if p[2] < 10 else "synth-val v10-18",   # forgetting shows up in v10-18
              "blur": "hard" if p[4] >= cuts[1] else ("mid" if p[4] >= cuts[0] else "easy")} for p in val_pool]

real_items = []
for f in REAL_FILES:
    with np.load(f, allow_pickle=True) as d:   # index arrays only after loading them ONCE (npz is lazy)
        meta = json.loads(str(d["meta"]))
        inputs, target, tert, fidx, aligned = d["inputs"], d["target"], d["tertile"], d["frame_idx"], d["aligned"]
    for i in np.flatnonzero(aligned):
        # Contrast stretch, as the deployed pipeline does (diffuqr.pipeline.module_input).
        real_items.append({"input": contrast_stretch(inputs[i]), "grid": target, "version": meta["version"],
                           "payload": meta["payload"], "split": f.stem, "blur": str(tert[i]),
                           "frame_idx": int(fidx[i])})
if REAL_LIMIT:
    order = {"blurriest": 0, "medium": 1, "sharpest": 2}
    real_items = sorted(real_items, key=lambda r: order[r["blur"]])[:REAL_LIMIT]
print("real eval frames:", len(real_items))

In [ ]:
# ---- Model, optimiser, EMA, resume from the Hub ------------------------------
import math, torch
from huggingface_hub import hf_hub_download
from diffuqr.diffusion import EMA, ModelConfig, ModuleDiffusion, count_params, make_train_scheduler

torch.manual_seed(SEED)
model_cfg = ModelConfig()
model = ModuleDiffusion(model_cfg).to(DEVICE)
ema = EMA(model, EMA_DECAY)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = make_train_scheduler()
USE_AMP = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
step = 0
print(f"params: {count_params(model) / 1e6:.2f}M")

ckpt_name = f"{RUN_NAME}/last.pt"
# Resume from the NEWER of: the Hub checkpoint (pushed every PUSH_EVERY) and the local one
# (saved every SAVE_EVERY; survives a kernel restart within the same Kaggle session).
ck = None
for src in ("hub", "local"):
    try:
        if src == "hub":
            path = hf_hub_download(MODEL_REPO, ckpt_name, token=HF_TOKEN, local_dir=str(WORK / "hub_ckpt"))
        else:
            path = CKPT_DIR / "last.pt"
            if not path.exists():
                raise FileNotFoundError(path)
        c = torch.load(path, map_location="cpu", weights_only=False)
        print(f"  {src} checkpoint: step {c['step']}")
        if ck is None or c["step"] > ck["step"]:
            ck, ck_src = c, src
    except Exception as e:
        print(f"  {src} checkpoint: none ({type(e).__name__})")
if ck is not None:
    model.load_state_dict(ck["model"]); ema.shadow.load_state_dict(ck["ema"]); opt.load_state_dict(ck["opt"])
    scaler.load_state_dict(ck["scaler"]); step = ck["step"]
    print(f"resumed {ckpt_name} from {ck_src} at step {step}")
    del ck, c
elif INIT_FROM:
    c = torch.load(hf_hub_download(MODEL_REPO, INIT_FROM, token=HF_TOKEN, local_dir=str(WORK / "hub_ckpt")),
                   map_location="cpu", weights_only=False)
    assert c.get("mode", "diffusion") == MODE, f"{INIT_FROM} was trained with mode={c.get('mode')}"
    # Fine-tune from the EMA weights (the ones evaluated); fresh optimiser and LR schedule, step 0.
    model.load_state_dict(c["ema"]); ema.shadow.load_state_dict(c["ema"])
    print(f"initialised from {INIT_FROM} (EMA weights at step {c['step']}); fresh optimiser, step 0")
    del c
else:
    print("no checkpoint to resume; starting from scratch")
ema.shadow.to(DEVICE)

# Multi-GPU: verify DataParallel on one real batch before committing to it; fall back to 1 GPU on any error.
train_model = model
if N_GPU > 1:
    from diffuqr.diffusion import collate, make_example, training_loss
    dp = torch.nn.DataParallel(model)
    probe = collate([make_example(p[0], p[1], p[2]) for p in train_pool[: max(2, N_GPU)]])
    probe = {k: v.to(DEVICE) for k, v in probe.items()}
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
            for m in ("diffusion", "direct"):
                training_loss(dp, probe, sched, mode=m, self_cond_p=1.0)
        train_model = dp
        print(f"DataParallel OK on {N_GPU} GPUs")
    except Exception as e:
        print(f"DataParallel failed ({type(e).__name__}: {e}); training on 1 GPU")
    del probe


def save_ckpt(push=False):
    path = CKPT_DIR / "last.pt"
    torch.save({"model": model.state_dict(), "ema": ema.shadow.state_dict(), "opt": opt.state_dict(),
                "scaler": scaler.state_dict(), "step": step, "model_cfg": model_cfg.__dict__,
                "mode": MODE, "run": RUN_NAME}, path)
    pushed = False
    if push and PUSH:
        api.upload_file(path_or_fileobj=str(path), path_in_repo=ckpt_name, repo_id=MODEL_REPO,
                        repo_type="model", commit_message=f"{RUN_NAME} step {step}")
        # Every push stores a new 409 MB version; squash so the repo keeps only the latest files.
        # (The model repo holds checkpoints/results only - never data.)
        api.super_squash_history(repo_id=MODEL_REPO, repo_type="model")
        pushed = True
    print(f"  checkpoint saved at step {step}" + (" and pushed" if pushed else " (local)"))

In [ ]:
# ---- Quick evaluation (used during training and at the end) ------------------
import pandas as pd
from diffuqr.diffusion import run_eval


def quick_eval(n_val=64, n_real=32, k_list=(1,), steps=10):
    """Synthetic held-out (hard blur third first) + real blurriest frames, EMA weights."""
    hard_first = sorted(val_items, key=lambda r: {"hard": 0, "mid": 1, "easy": 2}[r["blur"]])
    real_hard = [r for r in real_items if r["blur"] == "blurriest"] or real_items
    rows = run_eval(ema.shadow, hard_first[:n_val] + real_hard[:n_real], k_list, steps, DEVICE, mode=MODE)
    df = pd.DataFrame(rows)
    out = df.groupby(["split", "blur"]).agg(n=("module_err_k1", "size"), module_err=("module_err_k1", "median"),
                                           decode_k1=(f"verifier_k{k_list[0]}", lambda s: (s == "match").mean()))
    return out

In [ ]:
# ---- Training -----------------------------------------------------------------
import psutil
from torch.utils.data import DataLoader
from diffuqr.diffusion import PoolDataset, collate, training_loss

train_ds = PoolDataset([(p[0], p[1], p[2]) for p in train_pool], length=10 ** 9, p_fresh=P_FRESH,
                       bank_path=BANK_PATH, seed=SEED + step, sample_cfg=SAMPLE_CFG)
dl = DataLoader(train_ds, batch_size=BATCH, num_workers=NUM_WORKERS if ON_KAGGLE else 0, collate_fn=collate,
                pin_memory=DEVICE.type == "cuda", drop_last=True, persistent_workers=ON_KAGGLE and NUM_WORKERS > 0)


def lr_at(s):
    if s < WARMUP:
        return LR * (s + 1) / WARMUP
    return LR * 0.5 * (1 + math.cos(math.pi * min(1.0, (s - WARMUP) / max(1, TRAIN_STEPS - WARMUP))))


t_start, t_log, losses = time.time(), time.time(), []
train_model.train()
data_iter = iter(dl)
while step < TRAIN_STEPS:
    if (time.time() - t_start) / 3600 > MAX_HOURS:
        print("time budget reached; saving and stopping (re-run to resume)")
        break
    batch = {k: v.to(DEVICE, non_blocking=True) for k, v in next(data_iter).items()}
    for g in opt.param_groups:
        g["lr"] = lr_at(step)
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        loss = training_loss(train_model, batch, sched, mode=MODE)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt)
    scaler.update()
    ema.update(model)
    losses.append(loss.item())
    step += 1
    if step % LOG_EVERY == 0:
        dt = time.time() - t_log
        eta_h = (TRAIN_STEPS - step) / max(LOG_EVERY / dt, 1e-6) / 3600
        vm = psutil.virtual_memory()
        print(f"step {step:6d}  loss {np.mean(losses):.4f}  lr {lr_at(step):.2e}  {LOG_EVERY / dt:.2f} it/s"
              f"  ETA {eta_h:.1f} h  RAM {vm.used / 1e9:.1f}/{vm.total / 1e9:.0f} GB")
        losses, t_log = [], time.time()
    if step % PUSH_EVERY == 0:
        save_ckpt(push=True)
    elif step % SAVE_EVERY == 0:
        save_ckpt()
    if step % EVAL_EVERY == 0:
        print(quick_eval().to_string()); train_model.train()
if step % PUSH_EVERY != 0:
    save_ckpt(push=True)   # always push on stop (time budget or done) so the next session can resume
print(f"training stopped at step {step} after {(time.time() - t_start) / 60:.1f} min")

In [ ]:
# ---- Final evaluation: decode rate vs K (EMA weights) --------------------------
from diffuqr.modspace import baseline_read
from diffuqr.qrcodec import decode_soft
from diffuqr.qrgrid import modules_for_version, read_format

t0 = time.time()
rows = run_eval(ema.shadow, val_items + real_items, K_LIST, SAMPLE_STEPS, DEVICE, mode=MODE)
res = pd.DataFrame(rows)
print(f"evaluated {len(res)} items in {(time.time() - t0) / 60:.1f} min")

# Learning-free baseline on the SAME inputs: module-centre reader + soft RS (EC/mask read from the grid).
def baseline_status(it):
    n = modules_for_version(it["version"])
    bits, conf = baseline_read(it["input"], n, 8, 2)
    e, m, _ = read_format(bits, it["version"])
    r, _ = decode_soft(bits, conf, it["version"], e, m)
    return "match" if (r.ok and r.payload == it["payload"]) else ("wrong" if r.ok else "none")
res["baseline"] = [baseline_status(it) for it in val_items + real_items]

def rate(col):
    return lambda s: f"{100 * (s == 'match').mean():.0f}%"

agg = {"n": ("baseline", "size"), "baseline": ("baseline", rate("baseline")),
       "module_err_k1": ("module_err_k1", "median"), "module_err_post": ("module_err_post", "median")}
for k in K_LIST:
    agg[f"verifier_K{k}"] = (f"verifier_k{k}", rate(k))
for k in K_LIST:
    agg[f"posterior_K{k}"] = (f"posterior_k{k}", rate(k))
table = res.groupby(["split", "blur"]).agg(**agg)
wrong = {c: int((res[c] == "wrong").sum()) for c in res.columns if c.startswith(("verifier", "posterior", "baseline"))}
ms = res["ms_per_item_allK"].median()
print(table.to_string())
print("wrong decodes:", wrong)
print(f"sampling time: {ms:.0f} ms per frame for K={max(K_LIST)}, {SAMPLE_STEPS} DDIM steps")

In [ ]:
# ---- Save results (and push them next to the checkpoint) --------------------
out_csv = WORK / f"results_{RUN_NAME}.csv"
out_md = WORK / f"results_{RUN_NAME}.md"
res.to_csv(out_csv, index=False)
out_md.write_text(
    f"# {RUN_NAME} (mode={MODE}, step={step}, K={K_LIST}, DDIM steps={SAMPLE_STEPS})\n\n"
    + table.to_markdown() + f"\n\nWrong decodes: {wrong}\n\nSampling: {ms:.0f} ms/frame for K={max(K_LIST)}\n",
    encoding="utf-8")
if PUSH:
    for p in (out_csv, out_md):
        api.upload_file(path_or_fileobj=str(p), path_in_repo=f"{RUN_NAME}/{p.name}", repo_id=MODEL_REPO,
                        repo_type="model", commit_message=f"{RUN_NAME} results at step {step}")
print("saved", out_csv.name, out_md.name, "(pushed)" if PUSH else "")

In [ ]:
# ---- Qualitative: real blurriest frames (input | GT | P(dark) | errors) ------
import matplotlib.pyplot as plt
from diffuqr.diffusion import collate, make_example, sample, symbol_from_canvas

show = [r for r in real_items if r["blur"] == "blurriest"][:4] or real_items[:4]
if show:
    b = collate([make_example(r["input"], r["grid"], r["version"]) for r in show])
    p = sample(ema.shadow, {k: v.to(DEVICE) for k, v in b.items()}, K=max(K_LIST), steps=SAMPLE_STEPS,
               mode=MODE).cpu().numpy()
    fig, axes = plt.subplots(len(show), 4, figsize=(12, 3 * len(show)), squeeze=False)
    for i, r in enumerate(show):
        post = symbol_from_canvas(p[i], r["version"]).mean(0)
        for ax, im, title in zip(axes[i], [r["input"], 1 - r["grid"], 1 - post, (post > 0.5) != r["grid"]],
                                 ["blurry input", "ground truth", f"P(dark), K={max(K_LIST)}", "errors"]):
            ax.imshow(im, cmap="gray", interpolation="nearest"); ax.set_title(title); ax.axis("off")
    plt.tight_layout(); plt.savefig(WORK / f"qualitative_{RUN_NAME}.png", dpi=80); plt.show()